In [2]:
!pip install -q transformers datasets scikit-learn huggingface_hub

from datasets import load_dataset
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from huggingface_hub import login
import torch
import torch.nn as nn
import numpy as np
import gc

print("All imports done.")

All imports done.


In [3]:
dataset  = load_dataset("freococo/burmese-contextual-pragmatics")
train_ds = dataset["train"]

politeness_map = {
    "neutral":"neutral", "polite":"polite", "informal":"informal",
    "professional":"professional", "blunt":"blunt", "rude":"rude",
    "very_polite":"polite", "friendly":"informal", "religious":"polite",
    "impolite":"rude", "stern":"blunt", "sarcastic":"rude",
    "polite_but_firm":"polite"
}

positive = {"Warm","Sweet","Cheerful","Soft","Sincere","Friendly","Kind",
            "Caring","Gentle","Encouraging","Supportive","Peaceful","Grateful",
            "Romantic","Joyful","Hopeful","Welcoming","Trusting","Approving",
            "Complimentary","Amused","Excited","Moved","Nostalgic","Sympathetic",
            "Pitying","Curious","Playful","Tender","Awe-struck","Awe",
            "Grateful/Humble","Care"}
formal_t = {"Professional","Serious","Respectful","Humble","Stern","Firm",
            "Determined","Authoritative","Commanding","Objective","Deep",
            "Intense","Stern/Concerned"}
negative = {"Angry","Annoyed","Hostile","Aggressive","Harsh","Hateful",
            "Disgusted","Cold","Dismissive","Sarcastic","Impolite","Rude",
            "Blunt","Crude","Envious","Dangerous","Fed up","Dry","Indifferent"}
emotional= {"Urgent","Pleading","Dramatic","Sad","Distressed","Tired",
            "Exhausted","Panicked","Shocked","Surprised","Concerned",
            "Apologetic","Appologetic","Regretful","Emotional","Teasing",
            "Casual","Dazed","Confused","Alarmed","Overwhelmed","Startled",
            "Devastated","Worried","Nervous/Sincere","Sad/Soft","Sad/Sweet",
            "Distress","Painful","Sorrowful","Weak","Uncertain","Weary",
            "Hurry","Impatient","Relieved","Fearful","Scared","Frightened",
            "Grieved","Incredulous","Bored","Lazy","Sleepy","Stressed",
            "Exasperated","Frustrated","Mischievous","Suggestive","Spooky",
            "Whispering","Indirect","Exclamatory","Lazy/Neutral",
            "Casual/Sleepy","Cheerful/Tired","Sweet/Weary","Sad/Serious",
            "Soft/Vulnerable","Natural","Considerate","Faithful","Trusting"}

def coarsen_tone(t):
    if t in positive:  return "positive"
    if t in formal_t:  return "formal"
    if t in negative:  return "negative"
    if t in emotional: return "emotional"
    return "neutral"

power_map = {
    "equal":"equal",
    "inferior_to_superior":"inferior_to_superior",
    "superior_to_inferior":"superior_to_inferior",
    "any":"any",
    "customer_to_staff":"inferior_to_superior",
    "customer_to_seller":"inferior_to_superior",
    "passenger_to_driver":"inferior_to_superior",
    "patient_to_doctor":"inferior_to_superior",
}

def apply_all_labels(example):
    example["politeness_coarse"] = politeness_map.get(example["politeness"], "neutral")
    example["tone_coarse"]       = coarsen_tone(example["tone"])
    example["power_coarse"]      = power_map.get(example["power_relation"], "equal")
    example["register_label"]    = example["register"]
    return example

train_ds = train_ds.map(apply_all_labels)

le_pol  = LabelEncoder().fit(train_ds["politeness_coarse"])
le_reg  = LabelEncoder().fit(train_ds["register_label"])
le_pow  = LabelEncoder().fit(train_ds["power_coarse"])
le_tone = LabelEncoder().fit(train_ds["tone_coarse"])

print("Politeness:", list(le_pol.classes_))
print("Register:  ", list(le_reg.classes_))
print("Power:     ", list(le_pow.classes_))
print("Tone:      ", list(le_tone.classes_))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

burmese-conversational-pragmatics.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2200 [00:00<?, ? examples/s]

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

Politeness: [np.str_('blunt'), np.str_('informal'), np.str_('neutral'), np.str_('polite'), np.str_('professional'), np.str_('rude')]
Register:   [np.str_('colloquial'), np.str_('formal'), np.str_('slang'), np.str_('standard')]
Power:      [np.str_('any'), np.str_('equal'), np.str_('inferior_to_superior'), np.str_('superior_to_inferior')]
Tone:       [np.str_('emotional'), np.str_('formal'), np.str_('negative'), np.str_('neutral'), np.str_('positive')]


In [4]:
def encode_all_labels(example):
    example["label_pol"]  = int(le_pol.transform([example["politeness_coarse"]])[0])
    example["label_reg"]  = int(le_reg.transform([example["register_label"]])[0])
    example["label_pow"]  = int(le_pow.transform([example["power_coarse"]])[0])
    example["label_tone"] = int(le_tone.transform([example["tone_coarse"]])[0])
    return example

train_ds = train_ds.map(encode_all_labels)

split1     = train_ds.train_test_split(test_size=0.30, seed=42)
split2     = split1["test"].train_test_split(test_size=0.50, seed=42)
train_data = split1["train"]
val_data   = split2["train"]
test_data  = split2["test"]

print(f"Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")

def get_weights(data, label_col, n_classes):
    counts = Counter(data[label_col])
    total  = len(data)
    return torch.tensor(
        [total / (n_classes * counts[i]) for i in range(n_classes)],
        dtype=torch.float
    )

w_pol  = get_weights(train_data, "label_pol",  len(le_pol.classes_))
w_reg  = get_weights(train_data, "label_reg",  len(le_reg.classes_))
w_pow  = get_weights(train_data, "label_pow",  len(le_pow.classes_))
w_tone = get_weights(train_data, "label_tone", len(le_tone.classes_))

print("Weights pol: ", [round(x,3) for x in w_pol.tolist()])
print("Weights reg: ", [round(x,3) for x in w_reg.tolist()])
print("Weights pow: ", [round(x,3) for x in w_pow.tolist()])
print("Weights tone:", [round(x,3) for x in w_tone.tolist()])

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

Train: 1540  Val: 330  Test: 330
Weights pol:  [5.238, 0.976, 0.318, 0.817, 3.889, 6.111]
Weights reg:  [0.364, 4.01, 3.565, 1.39]
Weights pow:  [3.377, 0.33, 2.567, 3.5]
Weights tone: [1.149, 1.257, 4.738, 0.487, 0.936]


In [5]:
# ── Trainer + Metrics ─────────────────────────────────────────────────────────

class MultiTaskTrainer(Trainer):
    def __init__(self, w_reg, w_pow, w_tone, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.w_reg  = w_reg
        self.w_pow  = w_pow
        self.w_tone = w_tone

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(
            input_ids      = inputs["input_ids"],
            attention_mask = inputs["attention_mask"],
            label_reg      = labels[:, 0],
            label_pow      = labels[:, 1],
            label_tone     = labels[:, 2],
            w_reg          = self.w_reg,
            w_pow          = self.w_pow,
            w_tone         = self.w_tone,
        )
        return (outputs["loss"], outputs) if return_outputs else outputs["loss"]

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        labels = inputs.pop("labels")
        with torch.no_grad():
            outputs = model(
                input_ids      = inputs["input_ids"],
                attention_mask = inputs["attention_mask"],
                label_reg      = labels[:, 0],
                label_pow      = labels[:, 1],
                label_tone     = labels[:, 2],
                w_reg          = self.w_reg,
                w_pow          = self.w_pow,
                w_tone         = self.w_tone,
            )
        loss = outputs["loss"]
        logits = (outputs["logits_reg"], outputs["logits_pow"], outputs["logits_tone"])
        return loss, logits, labels

def compute_metrics_mt(eval_pred):
    logits_reg, logits_pow, logits_tone = eval_pred.predictions
    label_ids = eval_pred.label_ids
    label_reg  = label_ids[:, 0]
    label_pow  = label_ids[:, 1]
    label_tone = label_ids[:, 2]
    p_reg  = np.argmax(logits_reg,  axis=1)
    p_pow  = np.argmax(logits_pow,  axis=1)
    p_tone = np.argmax(logits_tone, axis=1)
    return {
        "macro_f1_reg":  round(f1_score(label_reg,  p_reg,  average="macro"), 4),
        "macro_f1_pow":  round(f1_score(label_pow,  p_pow,  average="macro"), 4),
        "macro_f1_tone": round(f1_score(label_tone, p_tone, average="macro"), 4),
        "macro_f1_mean": round(np.mean([
            f1_score(label_reg,  p_reg,  average="macro"),
            f1_score(label_pow,  p_pow,  average="macro"),
            f1_score(label_tone, p_tone, average="macro"),
        ]), 4),
    }

login()
print("Setup complete.")

Setup complete.


In [6]:
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

KEEP_COLS = ["input_ids", "attention_mask", "labels"]

def tokenize_multitask(data):
    def fn(example):
        text = (example["utterance"]
                + " </s> " + str(example["context"]).strip())
        enc = tokenizer(
            text,
            max_length=MAX_LENGTH, truncation=True, padding="max_length"
        )
        enc["labels"] = [
            example["label_reg"],
            example["label_pow"],
            example["label_tone"]
        ]
        return enc
    cols_to_remove = [c for c in data.column_names if c not in KEEP_COLS]
    out = data.map(fn, remove_columns=cols_to_remove)
    out.set_format("torch")
    return out

print("Tokenizing...")
tok_train_MT = tokenize_multitask(train_data)
tok_val_MT   = tokenize_multitask(val_data)
tok_test_MT  = tokenize_multitask(test_data)
print("Done.")
print("Sample keys:", list(tok_train_MT[0].keys()))
print("Sample labels:", tok_train_MT[0]["labels"])

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing...


Map:   0%|          | 0/1540 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Done.
Sample keys: ['input_ids', 'attention_mask', 'labels']
Sample labels: tensor([0, 1, 3])


In [7]:
class MultiTaskXLMR(nn.Module):
    def __init__(self, model_name, n_reg, n_pow, n_tone):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size

        self.head_reg  = nn.Linear(hidden, n_reg)
        self.head_pow  = nn.Linear(hidden, n_pow)
        self.head_tone = nn.Linear(hidden, n_tone)

    def forward(self, input_ids, attention_mask,
                labels=None,
                label_reg=None, label_pow=None, label_tone=None,
                w_reg=None, w_pow=None, w_tone=None):

        cls = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0, :]

        logits_reg  = self.head_reg(cls)
        logits_pow  = self.head_pow(cls)
        logits_tone = self.head_tone(cls)

        loss = None
        if label_reg is not None:
            loss_reg  = nn.CrossEntropyLoss(weight=w_reg.to(cls.device)  if w_reg  is not None else None)(logits_reg,  label_reg)
            loss_pow  = nn.CrossEntropyLoss(weight=w_pow.to(cls.device)  if w_pow  is not None else None)(logits_pow,  label_pow)
            loss_tone = nn.CrossEntropyLoss(weight=w_tone.to(cls.device) if w_tone is not None else None)(logits_tone, label_tone)
            loss = (loss_reg + loss_pow + loss_tone) / 3

        return {
            "loss": loss,
            "logits_reg":  logits_reg,
            "logits_pow":  logits_pow,
            "logits_tone": logits_tone,
        }

n_reg  = len(le_reg.classes_)
n_pow  = len(le_pow.classes_)
n_tone = len(le_tone.classes_)

print("Multi-task model defined.")
print(f"Heads — register: {n_reg} | power: {n_pow} | tone: {n_tone}")

Multi-task model defined.
Heads — register: 4 | power: 4 | tone: 5


In [8]:
HUB_REPO = "annasus10/xlmr-burmese-multitask"

args_MT = TrainingArguments(
    output_dir=HUB_REPO,
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=150,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1_mean",
    logging_steps=50,
    report_to="none",
    seed=42,
    push_to_hub=False,
    remove_unused_columns=False,
)

model_MT = MultiTaskXLMR(MODEL_NAME, n_reg, n_pow, n_tone)

trainer_MT = MultiTaskTrainer(
    w_reg=w_reg, w_pow=w_pow, w_tone=w_tone,
    model=model_MT,
    args=args_MT,
    train_dataset=tok_train_MT,
    eval_dataset=tok_val_MT,
    compute_metrics=compute_metrics_mt,
)

trainer_MT.train()
print("Multi-task training done.")

# ── Save to Google Drive immediately ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
save_dir = "/content/drive/MyDrive/multitask_model"
os.makedirs(save_dir, exist_ok=True)
torch.save(model_MT.state_dict(), f"{save_dir}/pytorch_model.bin")
torch.save({"n_reg": n_reg, "n_pow": n_pow, "n_tone": n_tone},
           f"{save_dir}/config.pt")
print(f"Saved to Google Drive at {save_dir}")
print("Multi-task training done.")

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,Macro F1 Reg,Macro F1 Pow,Macro F1 Tone,Macro F1 Mean
1,1.493215,1.303143,0.402900,0.311700,0.358500,0.357700
2,1.150520,1.215818,0.569300,0.430400,0.383400,0.461100
3,0.969999,0.827284,0.631400,0.659700,0.485600,0.592200
4,0.749714,0.810277,0.703800,0.611900,0.532400,0.616000
5,0.663311,0.750991,0.697400,0.701600,0.591700,0.663600
6,0.540120,0.720473,0.738000,0.669100,0.621800,0.676300
7,0.480075,0.719969,0.733000,0.696700,0.561700,0.663800
8,0.440600,0.744611,0.736500,0.712000,0.592400,0.680300


Multi-task training done.


ValueError: mount failed

In [9]:
# ── Save locally immediately — don't wait for Drive ──────────────────────────
import os, torch

os.makedirs("multitask_saved", exist_ok=True)
torch.save(model_MT.state_dict(), "multitask_saved/pytorch_model.bin")
torch.save({"n_reg": n_reg, "n_pow": n_pow, "n_tone": n_tone},
           "multitask_saved/config.pt")
print("Saved locally to multitask_saved/")

# ── Also download directly to your computer ──────────────────────────────────
from google.colab import files
files.download("multitask_saved/pytorch_model.bin")
files.download("multitask_saved/config.pt")
print("Download triggered.")

Saved locally to multitask_saved/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered.


In [10]:
preds_out = trainer_MT.predict(tok_test_MT)
logits_reg, logits_pow, logits_tone = preds_out.predictions

p_reg  = np.argmax(logits_reg,  axis=1)
p_pow  = np.argmax(logits_pow,  axis=1)
p_tone = np.argmax(logits_tone, axis=1)

true_reg  = preds_out.label_ids[:, 0]
true_pow  = preds_out.label_ids[:, 1]
true_tone = preds_out.label_ids[:, 2]

print("── Register ──")
print(classification_report(true_reg,  p_reg,  target_names=le_reg.classes_))

print("── Power ──")
print(classification_report(true_pow,  p_pow,  target_names=le_pow.classes_))

print("── Tone ──")
print(classification_report(true_tone, p_tone, target_names=le_tone.classes_))

print("\n── Summary ──")
print(f"Register macro F1:  {f1_score(true_reg,  p_reg,  average='macro'):.4f}  (old: 0.705)")
print(f"Power    macro F1:  {f1_score(true_pow,  p_pow,  average='macro'):.4f}  (old: 0.598)")
print(f"Tone     macro F1:  {f1_score(true_tone, p_tone, average='macro'):.4f}  (old: 0.613)")

── Register ──
              precision    recall  f1-score   support

  colloquial       0.92      0.82      0.87       222
      formal       0.74      0.67      0.70        21
       slang       0.73      0.76      0.75        29
    standard       0.55      0.79      0.65        58

    accuracy                           0.80       330
   macro avg       0.74      0.76      0.74       330
weighted avg       0.83      0.80      0.81       330

── Power ──
                      precision    recall  f1-score   support

                 any       0.41      0.57      0.48        30
               equal       0.92      0.80      0.86       241
inferior_to_superior       0.70      0.91      0.79        33
superior_to_inferior       0.49      0.69      0.57        26

            accuracy                           0.78       330
           macro avg       0.63      0.74      0.67       330
        weighted avg       0.82      0.78      0.79       330

── Tone ──
              precision    r

In [12]:
# ── Cell 8: Stage 4 with Multi-Task Predictions ───────────────────────────────

from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load Stage 3 politeness model from HF
print("Loading Stage 3 politeness model...")
model_pol = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-pragmatics-stage3-v2"
).to(device).eval()

reg_classes  = list(le_reg.classes_)
pow_classes  = list(le_pow.classes_)
tone_classes = list(le_tone.classes_)
pol_classes  = list(le_pol.classes_)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_MT.to(device).eval()

def predict_mt_single(utterance, context):
    text = utterance + " </s> " + str(context).strip()
    enc  = tokenizer(text, max_length=128, truncation=True,
                     padding="max_length", return_tensors="pt").to(device)
    with torch.no_grad():
        out = model_MT(input_ids=enc["input_ids"],
                       attention_mask=enc["attention_mask"])
    reg  = reg_classes[torch.argmax(out["logits_reg"],  dim=1).item()]
    pow_ = pow_classes[torch.argmax(out["logits_pow"],  dim=1).item()]
    tone = tone_classes[torch.argmax(out["logits_tone"], dim=1).item()]
    return reg, pow_, tone

def predict_pol_single(text):
    enc = tokenizer(text, max_length=128, truncation=True,
                    padding="max_length", return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model_pol(**enc).logits
    return pol_classes[torch.argmax(logits, dim=1).item()]

predictions_stage4_mt = []
true_labels = []

print(f"Running Stage 4 MT pipeline on {len(test_data)} examples...")

for i, example in enumerate(test_data):
    utterance   = example["utterance"]
    context     = str(example["context"]).strip()
    instruction = str(example["instruction"]).strip()

    # Multi-task → register, power, tone in one shot
    pred_reg, pred_pow, pred_tone = predict_mt_single(utterance, context)

    # Stage 3 politeness model
    text_pol = (f"[register: {pred_reg}] [power: {pred_pow}] [tone: {pred_tone}] "
                f"{utterance} </s> {context} </s> {instruction}")
    pred_pol = predict_pol_single(text_pol)

    predictions_stage4_mt.append(pred_pol)
    true_labels.append(pol_classes[example["label_pol"]])

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/330 done...")

# ── Results ───────────────────────────────────────────────────────────────────
CLASSES  = list(le_pol.classes_)
acc      = accuracy_score(true_labels, predictions_stage4_mt)
macro_f1 = f1_score(true_labels, predictions_stage4_mt,
                    average="macro", labels=CLASSES, zero_division=0)
w_f1     = f1_score(true_labels, predictions_stage4_mt,
                    average="weighted", labels=CLASSES, zero_division=0)

print("\n── Stage 4 MT (Multi-Task Pipeline) Results ─────────────────────")
print(f"Accuracy:    {acc:.4f}")
print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {w_f1:.4f}")
print()
print(classification_report(true_labels, predictions_stage4_mt,
                            labels=CLASSES, zero_division=0))

print("\n── Full Comparison ──────────────────────────────────────────────")
print(f"{'Model':<45} {'Macro F1':>10}")
print("-" * 57)
print(f"{'Stage 1 (utterance only)':<45} {'0.7014':>10}")
print(f"{'Stage 2 (+ context)':<45} {'0.7583':>10}")
print(f"{'Stage 3 (oracle metadata)':<45} {'0.7990':>10}")
print(f"{'Stage 4 original (independent)':<45} {'0.7020':>10}")
print(f"{'Stage 4 v2 (codependent chain)':<45} {'0.6957':>10}")
print(f"{'Stage 4 MT (multi-task)':<45} {macro_f1:>10.4f}")
print(f"{'ChatGPT o4-mini (zero-shot)':<45} {'0.6346':>10}")
print(f"{'Gemini 3 Fast (zero-shot)':<45} {'0.4929':>10}")

Device: cuda
Loading Stage 3 politeness model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Running Stage 4 MT pipeline on 330 examples...
  50/330 done...
  100/330 done...
  150/330 done...
  200/330 done...
  250/330 done...
  300/330 done...

── Stage 4 MT (Multi-Task Pipeline) Results ─────────────────────
Accuracy:    0.7727
Macro F1:    0.6806
Weighted F1: 0.7739

              precision    recall  f1-score   support

       blunt       0.64      0.50      0.56        14
    informal       0.77      0.80      0.78        59
     neutral       0.88      0.78      0.83       174
      polite       0.63      0.86      0.73        56
professional       0.81      0.87      0.84        15
        rude       0.36      0.33      0.35        12

    accuracy                           0.77       330
   macro avg       0.68      0.69      0.68       330
weighted avg       0.78      0.77      0.77       330


── Full Comparison ──────────────────────────────────────────────
Model                                           Macro F1
---------------------------------------------------